# Lexical Search using TF-IDF

## Sparse Term-Frequency Document Vectors for Research-Paper Abstracts

**Developer Role:** TF-IDF Developer (Lexical Search)

---

### Overview

This notebook implements **lexical representation** for research-paper search using **TF-IDF** (Term Frequency - Inverse Document Frequency), a classic sparse vector-space model.

TF-IDF has no notion of meaning: it only knows whether a word *literally* occurs in a document, weighted by how rare that word is across the whole corpus. Two abstracts describing the same idea with different vocabulary ("car" vs. "automobile") will **not** be considered similar by this model alone - that limitation is exactly what this arm of the case study is meant to measure, against Word2Vec's static semantic embeddings and BERT/SPECTER's contextual embeddings.

### What This Component Does

- Accepts a list of research-paper abstracts
- Generates a **TF-IDF document matrix** for the entire corpus
- Generates a **TF-IDF vector** for a new user query, using the SAME fitted vocabulary
- Returns embeddings as `scipy.sparse` matrices

### What This Component Does NOT Do

- Does **not** compute cosine similarity or any distance metric as part of the deliverable class
- Does **not** rank or retrieve documents
- Those responsibilities belong to a separate component of the case study
  (a small demo at the end of this notebook exercises them anyway, purely
  to show the embeddings work - it is explicitly **not** part of the class)

### Preprocessing Philosophy

Unlike BERT/SPECTER, TF-IDF relies entirely on **exact lexical matches between tokens**. That makes preprocessing far more important here than for a transformer model: every inflected/derived form of a word must be reduced to the same token, or the vectorizer treats "network", "networks" and "networking" as three unrelated, independently-weighted vocabulary columns. This notebook therefore performs the FULL classical NLP pipeline:

- Lowercasing
- Tokenization
- Removal of punctuation and standard English stop words
- POS-aware lemmatization (NLTK `WordNetLemmatizer`)

### Pipeline

```
Abstracts / Query
       |
Preprocessing (lowercase -> tokenize -> remove punctuation
               & stop words -> lemmatize)
       |
TfidfVectorizer (fit on the corpus, reused to transform queries)
       |
Sparse Embedding Vectors (scipy.sparse matrices)
```

### Position in the Overall Comparison

| Approach | Representation | Developer |
|----------|---------------|-----------|
| **TF-IDF** | **Lexical (exact word matching)** | **This notebook** |
| Word2Vec | Static semantic (fixed word vectors) | Developer 2 |
| SPECTER | Contextual scientific document embeddings | Developer 3 |


---
## Cell 1 — Install Dependencies

Install the required libraries:
- **scikit-learn**: Provides `TfidfVectorizer`, which builds the vocabulary and computes TF-IDF weights
- **nltk**: Tokenization, the English stop-word list, POS tagging and the WordNet lemmatizer
- **numpy**: Used to preview/inspect the sparse embeddings
- **pandas**: Loads the research-paper dataset from CSV

The `-q` flag suppresses verbose installation output.

In [1]:
!pip install -q scikit-learn nltk numpy pandas


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Cell 2 — Imports

| Library | Purpose |
|---------|---------|
| `re`, `string` | Built-ins for basic cleaning (HTML/URLs) and the punctuation set |
| `numpy` | Used to preview/inspect the sparse embeddings |
| `pandas` | Loads the research-paper dataset from CSV |
| `nltk` | Tokenizer, English stop-word list, POS tagger, WordNet lemmatizer |
| `TfidfVectorizer` | Builds the vocabulary and computes TF-IDF weights (scikit-learn) |

In [2]:
# ============================================================
# Cell 2 — Imports
# ============================================================

import re                                        # Built-in: basic text cleaning (HTML tags, URLs, whitespace)
import string                                    # Built-in: the punctuation character set

import numpy as np                               # Used only to preview/inspect the sparse embeddings
import pandas as pd                              # Loads the dataset from CSV

import nltk                                      # Classical NLP toolkit (tokenizer, stop words, lemmatizer)
from nltk.corpus import stopwords, wordnet       # English stop-word list + WordNet POS constants
from nltk.stem import WordNetLemmatizer          # Rule/lexicon-based lemmatizer
from nltk.tokenize import word_tokenize          # Splits a sentence into a list of word tokens

from sklearn.feature_extraction.text import TfidfVectorizer  # Builds the TF-IDF vocabulary/matrix

print("All libraries imported successfully!")


All libraries imported successfully!
